# 🔬 report6 — 검증 리포트: **Sionna RT 는 표적 RCS 를 모델링하지 않는다**

> **이 노트북 = 방법론 검증.** report2~5 가 딛고 선 가장 중요한 가정 하나를 실험으로 판정합니다.
> 결과는 우리에게 유리하지도, 불리하지도 않은 **사실**입니다 — 그리고 그 사실이 설계를 바꿉니다.

**문제 제기 (정당했습니다)**

> *"Sionna 시뮬레이션이 목적인데 표적 RCS·에코를 PO 로 계산해 넣으면, 그걸 'Sionna 로 검증했다'고 할 수 있나?"*

이 프로젝트는 표적 σ 를 **물리광학(PO)** 으로 구하고 Sionna RT 는 **환경(챔버 벽·다중경로·클러터)** 에만 씁니다.
그러니 위 질문은 급소를 찌릅니다. 가능한 답은 셋이었습니다:

1. RT 로도 표적 에코가 나오고 PO 와 일치한다 → PO 는 지름길이고, 하이브리드는 검증된다.
2. RT 로 나오지만 PO 와 다르다 → 둘 중 하나가 틀렸다.
3. **RT 는 애초에 표적 RCS 를 모델링하지 않는다** → 하이브리드는 선택이 아니라 **필수**이고, 검증은 다른 축에서 해야 한다.

처음엔 2번처럼 보였습니다(RT 에코가 PO 대비 −5 ~ −40 dB 로 시드마다 요동). **결론은 3번이었습니다.**

**3줄 결론**
1. RT 가 표적에서 만드는 경로는 **정반사(무한거울)** 와 **확산(산란계수 S 의 함수)** 뿐 — **둘 다 σ 의 함수가 아닙니다.**
2. 물리적으로 옳은 **PEC 금속구에서는 RT 가 경로를 하나도 만들지 못합니다**(크기·표본수 무관). 곡면의 정반사점을 못 잡기 때문입니다.
3. 그래서 **표적 = PO, 환경 = RT** 하이브리드는 편의가 아니라 **필연**입니다. 검증축은 **PO ↔ 해석해**·**PO ↔ 실측**으로 가야 합니다.

> 📄 상세 판정: [`docs/VERIFY_RT_VS_PO.md`](docs/VERIFY_RT_VS_PO.md) · 재현: `benchmark/verify_rt_no_rcs.py`

## 1. 실험 설계 — 무엇을 어떻게 비교했나

**비교량: 직접파 대비 표적에코의 진폭비.** 절대 보정(안테나 이득·송신전력)과 무관해서, **두 엔진을 공정하게 맞댈 수 있는 유일한 양**입니다.

| | 계산 방법 |
|---|---|
| **Sionna RT** | 광선추적 → 표적을 맞고 오는 경로들의 복소이득 합 ÷ 직접파(LOS) 이득 |
| **PO + 링크버짓** | 바이스태틱 레이더 방정식: ratio = L·√(σ/4π) / (R₁·R₂) |

**기하** (30×20×11 m 챔버, `src/bistatic_scene.py`): L = 15.07 m · R₁ = 18.75 m · R₂ = 18.61 m · β = 47.6° · f = 3.5 GHz
→ 표적에코가 있어야 할 지연 **τ = (R₁+R₂)/c = 124.6 ns**, 직접파 **50.3 ns**.

**교정 표적으로 금속구(r=0.3 m)를 골랐습니다** — σ = πr² 로 정답이 알려져 있고 바이스태틱 각에도 무관하니까요.

> ⚠️ **여기서 첫 번째 실수를 했습니다.** *"구는 어떤 각도에서도 정반사점이 있으니 RT 에 가장 유리하다"* 고 생각했는데,
> **SBR 솔버 기준으로는 정확히 반대**였습니다. 구는 RT 에 **가장 불리한** 표적입니다(§3). 이 오해가 진단을 한참 지연시켰습니다.

## 2. 결과 ① — RT 의 표적 에코에는 σ 가 들어 있지 않다

![no sigma](outputs/figures/report6_rt_no_sigma.png)

### (a) 정반사 = 무한거울
표적 자리에 **금속 평판**을 놓고 이등분선에 법선을 맞춰(정반사 조건 충족) 변 길이를 **0.2 → 4 m** 로 키웠습니다.
이론 σ 는 4πA²/λ² 이므로 **52 dB 증가**합니다. 그런데 RT 의 진폭비는:

| 평판 변 | 0.2 m | 0.5 m | 1.0 m | 2.0 m | 4.0 m |
|---|---|---|---|---|---|
| 이론 σ | +4.4 dBsm | +20.3 | +32.3 | +44.4 | +56.4 |
| **RT 진폭비** | **−7.91 dB** | **−7.91** | **−7.91** | **−7.91** | **−7.91** |

**산포 0.00 dB.** 그리고 그 값은 정확히 image-source(무한거울) 예측 20·log₁₀(L/(R₁+R₂)) = **−7.88 dB** 입니다.
→ Sionna 의 정반사 필드는 **무한평면 거울장**입니다(곡률·유한개구 보정 없음). **정반사 경로가 잡혀도 RCS 정보는 0입니다.**

### (b) 확산 = 물리가 아니라 노브(knob)
같은 금속구에서 **산란계수 S 만** 0.2 → 1.0 으로 바꿨습니다:

| S | 0.2 | 0.5 | 0.9 | 1.0 |
|---|---|---|---|---|
| RT 진폭비 | −56.6 dB | −46.8 | −41.7 | −40.8 |
| 이론(πr²) 대비 | −12.8 dB | −3.0 | +2.0 | **+3.0** |

**S 만으로 +15.8 dB 이동** — 정확히 **S² 법칙**입니다(20·log₁₀(1.0/0.2) = 14 dB, 차이는 S=0.2 에서 경로가 6개뿐이라 생기는 몬테카를로 잡음).

> **이론값에 맞추려면 S ≈ 0.85 가 필요합니다.** 즉 *"RT 로 PO 를 검증한다"* 는 건 **자유 파라미터 S 를 피팅하는 순환논법**입니다.
> 우리가 재는 건 드론의 RCS 가 아니라 **우리가 고른 노브** 입니다.

## 3. 결과 ② — 물리적으로 옳은 금속구에서는 경로가 아예 없다

![missing](outputs/figures/report6_rt_missing.png)

ITU `metal` 재질은 **scattering_coefficient = 0** 입니다(= PEC, 물리적으로 옳습니다). 그러면 확산 채널이 비어 있고 **정반사만** 가능한데 —

| | spp 100만 | spp 1600만 |
|---|---|---|
| 구 r = 0.3 m | **0개** | **0개** |
| 구 r = 1.0 m | **0개** | **0개** |
| 구 r = 3.0 m | **0개** | **0개** |

**크기를 10배 키우고 표본을 16배 늘려도 0개.** 즉 표본 부족이 아니라 **솔버의 구조적 한계**입니다.

**그런데 같은 자리의 평판은 표본 100만으로도 즉시 잡힙니다.** → 메쉬·재질·솔버는 **정상**입니다(직접파 이득도 λ/(4πL) 이론과 일치).

### 원인: 디스코볼 문제
곡면을 삼각형으로 쪼개면 **어떤 패싯도 자기 평면의 정반사점을 자기 삼각형 안에 품지 못합니다.**
세분해도 소용없습니다 — 면 법선의 각도 간격과 면 크기가 **함께** 줄어들어 비율이 그대로거든요.
즉 판별 변수는 **크기가 아니라 곡률**입니다.

> ⚠️ 단, *"곡면에서 정반사는 원리적으로 불가능"* 이라고까지 말하면 안 됩니다. 메쉬 자세를 무작위로 돌리면 **수 % 확률**로
> 유효 패싯이 나타나고, 그때는 Sionna 도 정반사 경로를 만듭니다. 다만 **잡혀도 진폭은 틀립니다**(단일 패싯이 '평판'으로 반사 → 구 이론 대비 +15 dB).
> 정확한 표현은 **"저확률 + 잡혀도 진폭 무의미"** 입니다.

## 4. 결과 ③ — RT 가 **맞히는** 것과, 우리가 **틀렸던** 것

![delay](outputs/figures/report6_rt_delay.png)

여기서 제 앞선 보고를 정정합니다. 처음엔 *"챔버에서 RT 가 PO 대비 −5 dB 로 일치한다"* 고 했는데 — **허수였습니다.**

챔버에서 우리 분류 규약(`표적 2 m 이내를 지나면 표적에코`)이 받아들인 경로 **9개**를 지연축에 그대로 찍어 보면:

- **τ = 124.1 ns, 1-bounce — 딱 1개만 진짜 표적 에코**입니다(기대값 124.6 ns 와 일치 ✔).
- 나머지 **8개는 τ = 182 ~ 223 ns 의 2·3-bounce 벽 반사**입니다. 드론 근처를 스쳤다는 이유로 삼켜진 겁니다.

그 8개를 함께 더해서 나온 게 '−5 dB 일치'였습니다. **좋아 보이는 숫자가 사실은 오분류의 산물**이었던 셈입니다.

> **그런데 이 그림은 동시에 RT 의 강점을 증명합니다** — 진짜 에코가 **정확히 (R₁+R₂)/c 자리에** 있습니다.
> **RT 의 기하(지연·도플러)는 정확합니다. 못 주는 건 진폭(σ)뿐입니다.**

## 5. 그러면 PO 는 믿을 수 있나 — 자기 검증

RT 를 비판했으니 PO 도 같은 잣대로 시험해야 공정합니다. 그런데 여기서 **우리 검증 근거 하나가 무너졌습니다.**

### (a) 평판 검증은 진단력이 0이다
![diagnostic](outputs/figures/report6_po_diagnostic.png)

report2 는 *"금속 평판에서 PO = 4πA²/λ² 이론과 0.00 dB 일치"* 를 검증 근거로 써 왔습니다. 그런데 —
**수직입사에서는 위상항이 사라집니다(P·û ≡ 0).** 그래서 PO 커널에 **일부러 버그를 심어도** 평판은 그대로 통과합니다.
다섯 가지 변종으로 실제 시험했습니다:

| 커널 변종 | 평판 σ 오차 | **구 σ 오차** |
|---|---|---|
| 정상 | 0 | +0.005 dB |
| 위상계수 2k → k | **0** | **+6.06 dB** |
| 위상 부호 반전 | **0** | **+0.005 dB** ⚠️ |
| obliquity (n̂·û) 제거 | **0** | **−29.81 dB** |
| 조명면 판정 제거 | **0** | **+6.02 dB** |

평판 오차는 **모든 변종에서 1.8e-15 dB**(기계 오차) — 문자 그대로 아무것도 검증하지 못합니다.

### ⚠️ 그런데 구도 못 잡는 버그가 하나 있습니다
**위상 부호 반전**은 구에서도 +0.005 dB 로 통과합니다. 우연이 아닙니다 — 부호를 뒤집으면 E → E\* 이고,
**σ ∝ |E|² 는 켤레에 불변**이라 *어떤 형상의 모노스태틱 RCS 시험으로도* 검출할 수 없습니다.

그래서 두 번째 시험을 추가했습니다: **표적을 시선 방향으로 옮기며 위상 기울기 dφ/dR 를 재는 것**입니다.
정상 커널은 +1.00, 2k→k 는 +0.50, **부호 반전은 −1.00** 으로 정확히 잡힙니다.

> 🔴 **이건 중요합니다.** 위상 부호는 **마이크로도플러의 도플러 부호**를 결정합니다(report3).
> RCS 숫자는 전부 멀쩡한 채로 도플러만 뒤집힐 수 있었다는 뜻입니다. **σ 검증만으로는 부족합니다.**

### (b) 구 ↔ 해석적 PO: 우리 커널은 맞다
![convergence](outputs/figures/report6_po_convergence.png)

구의 **해석적 PO** 적분과 대조하면 우리의 이산 PO 는 **모든 점간격에서 |Δ| ≤ 0.038 dB** 로 재현합니다(λ/4 에서도!).
즉 **커널은 맞습니다.** 다만 매끄러운 볼록 표적에는 미세구조가 없어, 구는 *실제 표적에 필요한 점간격* 에 대해서는 아무것도 말해주지 않습니다.

그래서 드론 5종의 **방위평균 σ** 를 점간격별로 쟀습니다(λ/30 기준 편차):

| | Matrice 4E | Mini 5 Pro | Mavic 4 Pro | Phantom 4 | **S1000+** |
|---|---|---|---|---|---|
| λ/7 의 편차 | −0.018 dB | −0.051 | −0.060 | −0.100 | **−0.295 dB** |

**단조롭고 항상 낮은 쪽**입니다 — 즉 λ/7 은 **계통적으로 과소평가**합니다. 크기가 클수록(S1000+) 편향이 큽니다.
실측 앵커링의 ±2~3 dB 에 비하면 작지만, **알려진 편향**으로 문서에 남겨야 합니다.

## 6. 널(null)에 대한 정정 — 위치는 믿고, 깊이는 인용하지 마라

![nulls](outputs/figures/report6_nulls.png)

폴라 RCS 에 원 중심까지 꽂히는 바늘(깊은 널)이 이상하다는 지적에서 시작했습니다. 검증 결과:

1. **통계적으로는 정상**입니다 — 코히런트 산란체의 σ 는 지수분포(레일리 페이딩)를 따르고, 측정 CDF 가 이론과 맞습니다.
2. **널의 위치는 이산화에 안정적**입니다(λ/7 ~ λ/20 에서 최저 10개 널의 방위가 동일).
   (λ/7 → λ/20 으로 정밀화해도 **어떤 널도 0.25°(방위 격자 한 칸) 이상 움직이지 않습니다.**)
3. **그러나 깊이는 신뢰할 수 없습니다** — 같은 10개 널의 깊이가 이산화에 따라 **0.2 ~ 10.7 dB 요동**하고 수렴하지 않습니다.
4. **실제 레이더는 그 널을 못 봅니다** — 최저 널이 **−64.1 dBsm → (100 MHz 대역평균) −45.1 → (+3° 각도창) −41.0 dBsm**.
   **그 바늘의 23 dB 는 순전히 허구**입니다. 반면 **로브 피크(−11.48 → −11.74)와 방위평균(−20.80 → −20.83 dBsm)은 불변** — 앵커링 수치는 안전합니다.

> → 그래서 report2 의 방위 패턴 그림은 **"레이더가 실제로 보는 값"**(대역평균 + 각도창)으로 그립니다.
> 그리고 **널 깊이 숫자는 어디에도 인용하지 않습니다.**

In [ ]:
# (선택 실행) 핵심 3실험 재현 — GPU 필요 (CUDA_VISIBLE_DEVICES=2)
# !cd .. && CUDA_VISIBLE_DEVICES=2 python benchmark/verify_rt_no_rcs.py
import json, os
d = json.load(open('outputs/rt_no_rcs_verify.json'))
print('[A] 평판 크기 스윕 — σ 가 52 dB 변하는 동안 RT 진폭비는?')
for r in d['A_plate']:
    print(f"    변 {r['side']:>3}m  σ={r['sigma_dbsm']:+6.1f} dBsm  →  RT {r['ratio_db']:+7.2f} dB")
print('\n[B] 구의 산란계수 S 스윕 — RT 는 σ 가 아니라 S 를 잰다')
for r in d['B_sphere_S']:
    print(f"    S={r['S']:.1f}  →  RT {r['ratio_db']:+7.2f} dB  (이론 대비 {r['dev_db']:+6.2f} dB, 경로 {r['n']}개)")
print('\n[C] PEC 금속구 — 경로가 하나도 없다')
for r in d['C_pec_sphere']:
    print(f"    r={r['r']}m  spp={r['spp']:>10,}  →  표적경로 {r['n_paths']}개")
print('\n[D] 챔버: 표적 근방 경로 %d개 중 진짜 표적에코는 %d개'
      % (d['D_chamber_paths']['n_near'], d['D_chamber_paths']['n_true']))

## 7. 판정 & 남은 리스크

![hybrid](outputs/figures/report6_hybrid.png)

### 하이브리드(표적=PO, 환경=RT)는 정당한가 — **그렇다. 단, 이유를 정확히 말해야 한다.**

정당한 이유는 **"Sionna RT 의 path solver 에 산란적분(PO) 단계가 없어 표적 σ 가 창발하지 않기 때문"** 입니다.
우리 설정 실수나 튜닝 부족이 아니라 **도구의 설계 범위**입니다.

**정당화에 쓰면 안 되는 근거** (적대적 검증에서 깨진 것들):

- ❌ *"3GPP 가 지지하는 표준 관행"* — 범주오류입니다. 3GPP ISAC 의 RCS 파라미터화는 **통계적 채널모델 안**의 이야기입니다.
- ❌ *"RCS 는 레이트레이싱에서 창발하지 않는다"* — **거짓**입니다. SBR(GO+PO)은 기하에서 RCS 를 계산해냅니다.
  참인 명제는 훨씬 좁습니다: **"산란적분 단계가 없는 전파용 레이트레이서(Sionna RT 포함)에서는 창발하지 않는다."**
- ❌ *"정공법이 아예 없다"* — Sionna 메인테이너가 지목한 **custom PathSolver** 경로가 존재합니다. 우리가 안 하는 이유는 **비용**이지 부재가 아닙니다.

### 남은 리스크 (정직하게)

| 리스크 | 상태 |
|---|---|
| **PO 의 절대 RCS 불확실도** | **모릅니다.** 기존 문서의 ±3 dB 는 유도가 깨졌습니다 — 재유도 전까지 인용 금지 |
| PO 점간격 λ/7 의 계통 편향 | −0.1 ~ −0.3 dB (λ/30 대비). 소폭이지만 알려진 편향 |
| 널 깊이 | **인용 불가** (이산화 의존) |
| 챔버 RT 경로예산 절단 | `max_num_paths_per_src` 기본 1M 에 챔버 총경로수가 점근 중 — 확인 필요 |
| 드론 오목부 다중반사 | PO 가 원리적으로 못 다룸 (알려진 모델 한계) |

### 다음 단계
1. **PO 절대 불확실도 재유도** — 자기차폐를 켤 때 불투명 depth-buffer 를 **내부 산란체에 적용하면 안 됩니다**(반투명 셸 가정과 모순).
2. `max_num_paths_per_src` 를 올려 챔버 클러터 통계가 바뀌는지 1회 확인.
3. (선택) **in-Sionna RCS**(custom PathSolver + TX=RX 1-bounce 에서 σ 추출) — 하이브리드가 이미 정당하므로 급하지 않습니다.

---

> **이 리포트의 값어치는 '우리가 옳았다'가 아니라 '무엇이 틀렸는지 알아냈다'에 있습니다.**
> 챔버의 −5 dB '일치'는 오분류였고, 평판 검증은 항등식이었으며, 널 깊이는 인용 불가였습니다.
> 셋 다 **적대적 검증이 없었으면 그대로 남았을 것**입니다.